<a href="https://colab.research.google.com/github/agemagician/ProtTrans/blob/master/Embedding/PyTorch/Advanced/ProtT5-XL-UniRef50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Important Notes:
1. ProtT5-XL-UniRef50 has both encoder and decoder, for feature extraction we only load and use the encoder part.
2. Loading only the encoder part, reduces the inference speed and the GPU memory requirements by half.
2. In order to use ProtT5-XL-UniRef50 encoder, you must install the latest huggingface transformers version from their GitHub repo.
3. If you are intersted in both the encoder and decoder, you should use T5Model rather than T5EncoderModel.

<h3>Extracting protein sequences' features using ProtT5-XL-UniRef50 pretrained-model</h3>

**1. Load necessry libraries including huggingface transformers**

In [1]:
!pip install -q SentencePiece transformers

In [2]:
import torch
from transformers import T5EncoderModel, T5Tokenizer
import re
import numpy as np
import gc

/home/cdchiang/miniconda3/envs/ProtTrans/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<b>2. Load the vocabulary and ProtT5-XL-UniRef50 Model<b>

In [3]:
tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False )

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [4]:
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")

In [5]:
tokenizer.save_pretrained("../models/prot_t5_xl_uniref50")
model.save_pretrained("../models/prot_t5_xl_uniref50")

<b>3. Load the model into the GPU if avilabile and switch to inference mode<b>

In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [7]:
model = model.to(device)
model = model.eval()

<b>4. Create or load sequences and map rarely occured amino acids (U,Z,O,B) to (X)<b>

In [16]:
sequences_Example = ["A E T T T T F F F F F Q Q","S K T Z P"]

In [17]:
sequences_Example = [re.sub(r"[UZOB]", "X", sequence) for sequence in sequences_Example]

<b>5. Tokenize, encode sequences and load it into the GPU if possibile<b>

In [18]:
ids = tokenizer.batch_encode_plus(sequences_Example, add_special_tokens=True, padding=True)

In [19]:
input_ids = torch.tensor(ids['input_ids']).to(device)
attention_mask = torch.tensor(ids['attention_mask']).to(device)

<b>6. Extracting sequences' features and load it into the CPU if needed<b>

In [20]:
with torch.no_grad():
    embedding = model(input_ids=input_ids,attention_mask=attention_mask)

In [21]:
embedding = embedding.last_hidden_state.cpu().numpy()

<b>7. Remove padding (\<pad\>) and special tokens (\</s\>) that is added by ProtT5-XL-UniRef50 model<b>

In [22]:
features = [] 
for seq_num in range(len(embedding)):
    seq_len = (attention_mask[seq_num] == 1).sum()
    seq_emd = embedding[seq_num][:seq_len-1]
    features.append(seq_emd)

In [23]:
print(features)

[array([[ 0.05521284, -0.3398663 ,  0.04084126, ...,  0.300471  ,
         0.00861673,  0.40233696],
       [ 0.19560342, -0.22779845, -0.21076463, ...,  0.15350133,
        -0.03121566,  0.22438651],
       [ 0.17737016, -0.07126054, -0.29989776, ..., -0.03928564,
         0.24806751,  0.01668222],
       ...,
       [-0.0349399 ,  0.07923291, -0.14820519, ...,  0.04409885,
        -0.07020636,  0.0555664 ],
       [ 0.09812708, -0.06797555, -0.13552603, ...,  0.06132533,
         0.12398481, -0.01396185],
       [ 0.14305219, -0.04111806,  0.07138339, ...,  0.05737103,
        -0.03629604, -0.04785092]], dtype=float32), array([[ 0.16555814, -0.09297126, -0.2260614 , ..., -0.07201436,
        -0.11815789,  0.15539686],
       [ 0.11265324, -0.12298611, -0.11732379, ...,  0.05776753,
        -0.30057907,  0.19830091],
       [ 0.3078941 , -0.104886  , -0.16040765, ..., -0.06531574,
         0.10468015, -0.07694176],
       [ 0.33790198, -0.20987596, -0.2879338 , ...,  0.03746894,
     

In [10]:
# import h5py
# f = h5py.File('protein_embeddings_test.h5', 'r')
# # f.keys()
# list(f.values())
# testset = f["WP_010896633_1_698-1042 [subseq from] type 2 lanthipeptide synthetase LanM [Halalkalibacterium halodurans]"]
# print(testset)

import h5py

def read_h5_file(file_path):
    with h5py.File(file_path, 'r') as h5_file:
        # List all groups
        keys = list(h5_file.keys())
        # print("Keys in the HDF5 file:", keys)
        
        key2mu = {}
        # Iterate through the file
        for key in keys:
            data = h5_file[key][:]
            new_key = key.replace("_", ".", 2)
            key2mu[new_key[:14]] = data
            # print(f"Data for {new_key}: {data.shape}")
            # If the data is too large, avoid printing the entire array
            # Instead, you can print a summary or first few elements
            # print(data)
        
        return key2mu

# Example usage
file_path = 'protein_embeddings_test.h5'
read_h5_file(file_path)

{'WP.010896633.1': array([ 0.04564082,  0.03313893, -0.00400343, ..., -0.03806341,
         0.04440143,  0.03331162], dtype=float32),
 'WP.053432182.1': array([ 0.04570779,  0.03238524, -0.00483071, ..., -0.03736959,
         0.04426986,  0.03362579], dtype=float32)}